# 외식업 10종: 공연 활동과 매출 관계 비교
한식·카페 2종에서 공개자료의 **외식업 분류 10종 전체**로 범위를 넓힌다. 지역과 기간은 종로구·마포구, 2023~2025년 12분기를 유지한다. 업종은 매출 크기나 회귀 유의성으로 골라내지 않았다. 기존 2종 결과를 확인한 뒤 범위를 확장했으므로 이 분석은 **탐색적 확장**이다.

500m 관측 등록 공연 수를 기본으로, 총매출 로그와 21~24시 매출 비중을 분석한다. 업종마다 상권 고정효과·연도분기 고정효과를 허용하고, 같은 상권의 여러 업종과 시계열을 함께 묶은 군집 공분산으로 업종 간 차이를 검정한다.

업종별 기울기 20개에는 BH FDR 보정값을, 두 결과변수의 업종 기울기 동일성 검정에는 Holm 보정값을 함께 보고한다. 300/800m, 업종별 12분기 균형표본, 점포 수 통제는 민감도 분석이다. 결과가 유의한 모형을 골라 최종모형으로 바꾸지 않는다.

**실행:** 08번 패널과 계산 대조용 `reports/20261005/05_interaction_results.csv`가 필요하다. 네트워크 요청과 원자료 수정은 하지 않는다.


In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import os
for name in ['OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'OMP_NUM_THREADS']:
    os.environ[name] = '1'
import json, hashlib, platform
import numpy as np
import pandas as pd
from scipy import linalg, stats
from statsmodels.stats.multitest import multipletests
from IPython.display import display, Markdown

ROOT = PROJECT_ROOT
DATA, REPORTS = project_path('dataset/analysis'), project_path('reports/20261005')
FIGURES = REPORTS / 'figures'
INPUT = DATA / 'food_industry_panel.csv'
assert INPUT.exists(), '먼저 08_외식업_범위확장과_데이터셋.ipynb를 실행하세요.'
input_hash = hashlib.sha256(INPUT.read_bytes()).hexdigest()
scope_metadata = project_json_loads((REPORTS / '08_food_scope_metadata.json').read_text(encoding='utf-8'))
assert scope_metadata['panel_sha256'] == input_hash, '08번 메타데이터와 패널 해시가 다릅니다.'
panel = pd.read_csv(INPUT, dtype={'area_id':str, 'industry_id':str, 'quarter':str})
KEY = ['area_id','industry_id','quarter']
assert not panel.duplicated(KEY).any()
INDUSTRIES = dict(panel[['industry_id','industry']].drop_duplicates().sort_values('industry_id').to_numpy())
assert set(INDUSTRIES) == {f'CS100{i:03d}' for i in range(1,11)}
OUTCOMES = {'log_sales':'총매출 log(1+원)', 'night_share_pct':'21~24시 매출 비중(%p)'}
panel['entity'] = panel['area_id'] + '_' + panel['industry_id']
panel['log_sales'] = np.log1p(panel['total_sales'].where(panel['total_sales'].ge(0)))
panel['night_share_pct'] = (100*panel['night_sales']/panel['total_sales']).where(
    panel['total_sales'].gt(0) & panel['night_sales'].ge(0) & panel['night_sales'].le(panel['total_sales']))
panel['log_stores'] = np.log1p(panel['store_count'].where(panel['store_count'].ge(0)))
pd.set_option('display.max_rows', 25)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
display(panel.groupby('industry', observed=True).agg(
    패널행수=('quarter','size'), 매출관측=('total_sales','count'), 야간비중관측=('night_share_pct','count')))


## 1. 표본과 고정효과 계산
각 결과변수에 필요한 값이 제공된 행만 사용하고, 상권×업종이 최소 2분기에 관측되어야 한다. 업종별 참여 상권이 20개 미만이거나 고정효과를 제거한 공연 활동에 변화가 없으면 해당 모형은 추정하지 않는다. 20개 기준은 검정력을 보장하는 기준이 아니다.

큰 더미 행렬을 한꺼번에 만들지 않고 FWL 정리를 이용한다. 업종마다 상권 평균을 먼저 제거한 뒤, **상권 평균을 제거한 분기 더미**에 다시 투영해 분기 효과를 제거한다. 불균형패널에서 단순히 상권 평균과 분기 평균을 한 번씩 빼는 계산과 다르다. 이 잔차화는 명시적 고정효과 회귀와 같은 계수·잔차를 만든다.


In [ ]:
def prepare_frame(outcome, radius=500, balanced=False, stores=False, industry_ids=None, source=None):
    source = panel if source is None else source
    frame = source if industry_ids is None else source.loc[source['industry_id'].isin(industry_ids)]
    frame = frame.copy()
    frame['log_activity'] = np.log1p(frame[f'known_performance_count_{radius}m'])
    required = [outcome, 'log_activity', *KEY] + (['log_stores'] if stores else [])
    frame = frame.dropna(subset=required)
    n = frame.groupby('entity')['quarter'].transform('nunique')
    return frame.loc[n.eq(12) if balanced else n.ge(2)].sort_values(KEY).copy()

def absorb_effects(frame, columns):
    target = frame[columns].astype(float)
    target_dm = target - target.groupby(frame['area_id']).transform('mean')
    quarter = pd.get_dummies(frame['quarter'], dtype=float)
    quarter_dm = quarter - quarter.groupby(frame['area_id']).transform('mean')
    fit, _, rank_quarter, _ = linalg.lstsq(
        quarter_dm.to_numpy(), target_dm.to_numpy(), cond=1e-10, lapack_driver='gelsd')
    residual = target_dm.to_numpy() - quarter_dm.to_numpy() @ fit
    rank_nuisance = frame['area_id'].nunique() + int(rank_quarter)
    return residual, rank_nuisance


## 2. 상권 군집 공분산
업종별 회귀계수는 각 업종의 분석과 같지만, 같은 상권에 속한 서로 다른 업종의 오차가 함께 움직일 수 있다. 상권별 score를 합쳐 공분산을 계산하므로 이러한 업종 간 공분산을 포함한다.

소표본 보정은 G/(G−1) × (N−1)/(N−K)를 사용한다. K에는 제거한 고정효과의 rank와 모든 업종별 기울기를 포함한다. 개별 계수는 자유도 G−1인 t 추론, 동일성 검정은 상권 군집 Wald F 추론을 사용한다. 인접한 서로 다른 상권 사이의 공간 상관까지 해결하는 방식은 아니다.


In [ ]:
def fit_food_fe(frame, outcome, stores=False):
    xcols = ['log_activity'] + (['log_stores'] if stores else [])
    blocks, eligibility, offset = [], [], 0
    for industry_id in INDUSTRIES:
        part = frame.loc[frame['industry_id'].eq(industry_id)].copy()
        areas = part['area_id'].nunique()
        if areas < 20:
            eligibility.append(dict(industry_id=industry_id, status='too_few_clusters',
                                    nobs=len(part), area_clusters=areas))
            continue
        residual, nuisance_rank = absorb_effects(part, [outcome, *xcols])
        y, x = residual[:,0], residual[:,1:]
        rank_x = np.linalg.matrix_rank(x)
        sd_x = float(x[:,0].std())
        status = 'estimated' if rank_x == len(xcols) and sd_x > 1e-10 else 'no_identifiable_within_change'
        eligibility.append(dict(industry_id=industry_id, status=status, nobs=len(part),
            area_clusters=areas, within_activity_sd=sd_x, nuisance_rank=nuisance_rank,
            no_raw_activity_change_areas=int(part.groupby('area_id')['log_activity'].nunique().le(1).sum())))
        if status != 'estimated':
            continue
        inv = np.linalg.inv(x.T @ x)
        beta = inv @ (x.T @ y)
        blocks.append(dict(industry_id=industry_id, frame=part, x=x, y=y, beta=beta,
                           residual=y-x@beta, bread=inv, nuisance_rank=nuisance_rank, offset=offset))
        offset += len(xcols)
    assert len(blocks) >= 2, '추정 가능한 업종이 2개 미만입니다.'
    area_ids = sorted(set().union(*(set(b['frame']['area_id']) for b in blocks)))
    area_map = {a:i for i,a in enumerate(area_ids)}
    n = sum(len(b['frame']) for b in blocks)
    k = sum(b['nuisance_rank'] + len(xcols) for b in blocks)
    g = len(area_ids)
    assert n > k and g > 1
    scores = np.zeros((g, offset))
    for b in blocks:
        codes = b['frame']['area_id'].map(area_map).to_numpy()
        sl = slice(b['offset'], b['offset']+len(xcols))
        np.add.at(scores[:,sl], codes, b['x']*b['residual'][:,None])
    bread = linalg.block_diag(*(b['bread'] for b in blocks))
    correction = g/(g-1) * (n-1)/(n-k)
    cov = correction * bread @ (scores.T @ scores) @ bread
    cov = (cov + cov.T)/2
    beta = np.concatenate([b['beta'] for b in blocks])
    assert np.isfinite(beta).all() and np.isfinite(cov).all() and (np.diag(cov)>0).all()
    return dict(blocks=blocks, beta=beta, cov=cov, nobs=n, clusters=g, rank=k,
                df=g-1, correction=correction, eligibility=eligibility)


## 3. 동결한 두 업종 기준선으로 계산 검증
최신 재수집 패널의 값은 과거 05번 결과와 달라질 수 있다. 따라서 05번 실행에 사용한 **과거 입력과 결과를 함께 동결한 스냅샷**으로 계산법을 검증한다. 동결 입력의 SHA-256을 확인한 뒤 기본·점포 수 통제 모형의 한식·카페·차이 계수, 군집 표준오차, 신뢰구간, p값을 대조한다. 다음 절부터는 최신 08번의 10종 패널을 사용한다.

In [ ]:
BASELINE = DATA / 'baseline_two_industry'
baseline_manifest = project_json_loads((BASELINE / 'snapshot_manifest.json').read_text(encoding='utf-8'))
for record in baseline_manifest['files']:
    assert hashlib.sha256((project_path(record['snapshot_path'])).read_bytes()).hexdigest() == record['sha256']
baseline_input = BASELINE / 'performance_sales_panel.csv'
baseline_input_hash = hashlib.sha256(baseline_input.read_bytes()).hexdigest()
baseline_panel = pd.read_csv(baseline_input, dtype={name: str for name in KEY})
baseline_panel['entity'] = baseline_panel['area_id'] + '_' + baseline_panel['industry_id']
baseline_panel['log_sales'] = np.log1p(baseline_panel['total_sales'].where(baseline_panel['total_sales'].ge(0)))
baseline_panel['night_share_pct'] = (100*baseline_panel['night_sales']/baseline_panel['total_sales']).where(
    baseline_panel['total_sales'].gt(0) & baseline_panel['night_sales'].ge(0)
    & baseline_panel['night_sales'].le(baseline_panel['total_sales']))
baseline_panel['log_stores'] = np.log1p(baseline_panel['store_count'].where(baseline_panel['store_count'].ge(0)))
reference_file = BASELINE / '05_interaction_results.csv'
reference_hash = hashlib.sha256(reference_file.read_bytes()).hexdigest()
old_results = pd.read_csv(reference_file)
verification = []
for scenario, stores in [('main_500',False),('stores_500',True)]:
    for outcome in OUTCOMES:
        frame = prepare_frame(outcome, stores=stores, industry_ids=['CS100001','CS100010'], source=baseline_panel)
        model = fit_food_fe(frame, outcome, stores)
        positions = {b['industry_id']:b['offset'] for b in model['blocks']}
        for label, weights in [('한식음식점',{'CS100001':1}),
                               ('커피음료',{'CS100010':1}),
                               ('카페 - 한식 차이',{'CS100001':-1,'CS100010':1})]:
            c = np.zeros(len(model['beta']))
            for industry_id, value in weights.items():
                c[positions[industry_id]] = value
            coef = float(c@model['beta'])
            se = float(np.sqrt(c@model['cov']@c))
            critical = stats.t.ppf(.975, model['df'])
            values = dict(coefficient=coef, standard_error=se, ci_low=coef-critical*se,
                          ci_high=coef+critical*se, p_value=2*stats.t.sf(abs(coef/se),model['df']))
            old = old_results.loc[old_results['scenario'].eq(scenario)
                & old_results['outcome'].eq(outcome) & old_results['contrast'].eq(label)].iloc[0]
            delta = max(abs(values[name]-old[name]) for name in values)
            assert delta < 1e-7, (scenario,outcome,label,delta)
            verification.append(dict(scenario=scenario,outcome=outcome,contrast=label,
                                      max_absolute_difference=delta,passed=True))
display(pd.DataFrame(verification))


## 4. 10종 기울기와 전체 업종 차이
업종별 기울기가 0과 다른지와 업종끼리 기울기가 서로 다른지는 별개의 질문이다. 전체 검정은 '추정 가능한 모든 업종의 기울기가 같다'를 귀무가설로 둔다. 일부 업종이 제외되면 검정에 실제 포함된 업종 수를 표시한다. 모든 업종의 공통 계수가 0이라는 검정은 아니다.


In [ ]:
SCENARIOS = [('main_500',500,False,False,'기본 500m'),
             ('radius_300',300,False,False,'거리 300m'),
             ('radius_800',800,False,False,'거리 800m'),
             ('balanced_500',500,True,False,'업종별 12분기 균형'),
             ('stores_500',500,False,True,'점포 수 통제')]
effect_rows, joint_rows, sample_rows, covariance_rows, eligibility_rows = [], [], [], [], []
model_objects = {}
for scenario, radius, balanced, stores, label in SCENARIOS:
    for outcome in OUTCOMES:
        frame = prepare_frame(outcome, radius, balanced, stores)
        m = fit_food_fe(frame,outcome,stores)
        model_objects[(scenario,outcome)] = m
        indices = [b['offset'] for b in m['blocks']]
        slope_beta = m['beta'][indices]
        slope_cov = m['cov'][np.ix_(indices,indices)]
        critical = stats.t.ppf(.975,m['df'])
        for index,b in enumerate(m['blocks']):
            coef, se = float(slope_beta[index]), float(np.sqrt(slope_cov[index,index]))
            effect_rows.append(dict(scenario=scenario,outcome=outcome,radius_m=radius,
                industry_id=b['industry_id'],industry=INDUSTRIES[b['industry_id']],
                coefficient=coef,standard_error=se,ci_low=coef-critical*se,ci_high=coef+critical*se,
                p_value=2*stats.t.sf(abs(coef/se),m['df']),nobs_industry=len(b['frame']),
                area_clusters_industry=b['frame']['area_id'].nunique(),
                nobs_total=m['nobs'],area_clusters_total=m['clusters']))
        contrasts = np.eye(len(indices))[1:] - np.eye(len(indices))[0]
        d, v = contrasts@slope_beta, contrasts@slope_cov@contrasts.T
        rank = np.linalg.matrix_rank(v)
        statistic = float(d@np.linalg.solve(v,d)/len(d)) if rank == len(d) else np.nan
        joint_rows.append(dict(scenario=scenario,outcome=outcome,estimated_industries=len(indices),
            status='estimated' if np.isfinite(statistic) else 'singular_contrast_covariance',
            f_statistic=statistic,df_num=len(d),df_denom=m['df'],
            p_value=stats.f.sf(statistic,len(d),m['df']) if np.isfinite(statistic) else np.nan))
        sample_rows.append(dict(scenario=scenario,label=label,outcome=outcome,nobs_total=m['nobs'],
            area_clusters_total=m['clusters'],estimated_industries=len(indices),design_rank=m['rank'],
            residual_df=m['nobs']-m['rank'],inference_df=m['df'],cluster_correction=m['correction']))
        for row in m['eligibility']:
            eligibility_rows.append(dict(scenario=scenario,outcome=outcome,**row))
        for i,b in enumerate(m['blocks']):
            for j,b2 in enumerate(m['blocks']):
                covariance_rows.append(dict(scenario=scenario,outcome=outcome,industry_id_a=b['industry_id'],
                                            industry_id_b=b2['industry_id'],covariance=slope_cov[i,j]))
effects, joint = pd.DataFrame(effect_rows), pd.DataFrame(joint_rows)
effects['p_fdr_primary'] = np.nan
mask = effects['scenario'].eq('main_500')
effects.loc[mask,'p_fdr_primary'] = multipletests(effects.loc[mask,'p_value'],method='fdr_bh')[1]
joint['p_holm_primary'] = np.nan
mask = joint['scenario'].eq('main_500') & joint['p_value'].notna()
joint.loc[mask,'p_holm_primary'] = multipletests(joint.loc[mask,'p_value'],method='holm')[1]
display(joint.loc[joint['scenario'].eq('main_500')])
display(pd.DataFrame(sample_rows).query("outcome == 'log_sales'"))


## 5. 기본 결과표
범위를 넓혀 결과를 살펴본 탐색 분석이며, 보정 p값만으로 인과관계나 중요한 경제적 크기가 보장되지 않는다. 95% 신뢰구간은 개별 구간으로 다중비교 동시 구간이 아니다. 업종마다 관측 상권과 기간 구성이 달라 표본 구성의 차이도 함께 읽는다.


In [ ]:
display(effects.loc[effects['scenario'].eq('main_500'),
    ['outcome','industry','coefficient','ci_low','ci_high','p_value','p_fdr_primary',
     'nobs_industry','area_clusters_industry']].sort_values(['outcome','industry']))
display(pd.DataFrame(eligibility_rows).query("scenario == 'main_500' and outcome == 'log_sales'")
        .merge(pd.DataFrame(INDUSTRIES.items(),columns=['industry_id','industry']),on='industry_id'))


## 6. 업종별 관계 시각화
업종 순서는 원자료 코드 순서이며 p값이나 계수 크기로 정렬하지 않았다. 아래의 좌우 두 그림은 단위가 다르므로 가로축 길이를 서로 비교하지 않는다. 총매출은 로그, 야간 비중은 퍼센트포인트다.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
font_names = {f.name for f in fm.fontManager.ttflist}
plt.rcParams['font.family'] = next((f for f in ['Malgun Gothic','NanumGothic','AppleGothic'] if f in font_names),'DejaVu Sans')
plt.rcParams.update({'axes.unicode_minus':False,'font.size':10,'figure.dpi':110})
fig, axes = plt.subplots(1,2,figsize=(13,6.6))
for ax,outcome in zip(axes,OUTCOMES):
    part = effects.loc[effects['scenario'].eq('main_500') & effects['outcome'].eq(outcome)].sort_values('industry_id')
    for i,row in enumerate(part.itertuples()):
        ax.errorbar(row.coefficient,i,xerr=[[row.coefficient-row.ci_low],[row.ci_high-row.coefficient]],
                    fmt='o',capsize=3,color='#23638C' if row.p_fdr_primary<.05 else '#86929D')
    ax.set_yticks(range(len(part)),part['industry'])
    ax.axvline(0,color='#666666',ls='--',lw=1); ax.invert_yaxis()
    ax.set_title(OUTCOMES[outcome]); ax.set_xlabel('log(1+관측 공연 수)의 기울기와 95% 구간')
    ax.grid(axis='x',alpha=.2)
fig.suptitle('외식업 10종 · 500m 기본모형\n파란색: 20개 탐색 검정의 BH FDR < 0.05',fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES/'09_외식업10종_기본결과.png',dpi=160,bbox_inches='tight')
plt.show(); plt.close(fig)


## 7. 거리·표본·점포 수에 따른 변화
큰 계수나 작은 p값이 나온 설정만 고르지 않고 모든 민감도 결과를 함께 제시한다. 균형표본은 **각 업종 안에서** 12분기 매출이 제공되는 상권을 고른다. 10종 모두가 같은 상권에서 제공되어야 한다는 조건은 쓰지 않는다.

점포 수 통제 모형에는 원자료에서 매출이 양수인데 점포 수가 0인 행도 포함된다. 이러한 자료 간 불일치는 별도 한계이며, 점포 수를 참값으로 보정한 모형을 의미하지 않는다.


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,8.5))
colors = ['#245F87','#58A79B','#C8813E','#8865A4','#929292']
for ax,outcome in zip(axes,OUTCOMES):
    for j,(scenario,_,_,_,label) in enumerate(SCENARIOS):
        part = effects.loc[effects['scenario'].eq(scenario) & effects['outcome'].eq(outcome)]
        for row in part.itertuples():
            i = list(INDUSTRIES).index(row.industry_id)
            ax.errorbar(row.coefficient,i+(j-2)*.13,
                xerr=[[row.coefficient-row.ci_low],[row.ci_high-row.coefficient]],
                fmt='o',markersize=3,capsize=2,color=colors[j],alpha=.8)
        ax.plot([],[],color=colors[j],marker='o',label=label)
    ax.set_yticks(range(10),list(INDUSTRIES.values()))
    ax.axvline(0,color='#555555',ls='--',lw=1); ax.invert_yaxis()
    ax.set_title(OUTCOMES[outcome]); ax.set_xlabel('기울기 및 개별 95% 구간')
    ax.grid(axis='y',alpha=.15)
axes[1].legend(loc='upper center',bbox_to_anchor=(.5,-.1),ncol=2,frameon=False)
fig.suptitle('외식업별 민감도 분석 · 전체 표본과 업종별 균형표본 비교')
fig.tight_layout()
fig.savefig(FIGURES/'09_외식업10종_민감도.png',dpi=160,bbox_inches='tight')
plt.show(); plt.close(fig)


## 8. 공연 10개→20개 비교 시나리오
log(1+공연 수)의 차이는 log(21/11)이다. 역변환한 총매출 지표는 조건부 (총매출+1)의 **기하평균 상대차이**이며 산술평균 매출 변화율이나 개입 효과가 아니다. 야간 비중은 퍼센트포인트 차이다. 전체 범위에 10·20이 포함되어도 동일 상권에서 그 변화를 실제 관측했다는 뜻은 아니다.

표의 `areas_spanning_10_to_20`은 관측 최솟값이 10 이하이고 최댓값이 20 이상인 상권 수다. 정확히 10개·20개인 두 분기를 관측했거나 인과 비교가 가능하다는 의미는 아니다.


In [ ]:
scenario_rows = []
for row in effects.loc[effects['scenario'].eq('main_500')].itertuples():
    values = np.array([row.coefficient,row.ci_low,row.ci_high]) * np.log(21/11)
    if row.outcome == 'log_sales':
        values = 100*np.expm1(values)
        unit = '(총매출+1) 조건부 기하평균 상대차이 %'
    else:
        unit = '야간 매출 비중 차이 %p'
    scenario_rows.append(dict(industry_id=row.industry_id,industry=row.industry,outcome=row.outcome,
        estimate=values[0],ci_low=values[1],ci_high=values[2],unit=unit))
changes = pd.DataFrame(scenario_rows)
display(changes)

# 같은 분석 표본에서 공연 수 범위를 확인한다.
support_rows = []
for outcome in OUTCOMES:
    for b in model_objects[('main_500',outcome)]['blocks']:
        counts = b['frame']['known_performance_count_500m']
        ranges = b['frame'].groupby('area_id')['known_performance_count_500m'].agg(['min','max'])
        support_rows.append(dict(outcome=outcome,industry_id=b['industry_id'],industry=INDUSTRIES[b['industry_id']],
            min_count=int(counts.min()),median_count=float(counts.median()),max_count=int(counts.max()),
            areas_spanning_10_to_20=int((ranges['min'].le(10)&ranges['max'].ge(20)).sum()),
            area_clusters=len(ranges)))
activity_support = pd.DataFrame(support_rows)
display(activity_support)


## 9. 결과 저장과 해석
원자료 미제공 매출을 0으로 채우지 않았다. 업종별 상권 매출 합계는 상권의 중첩을 제거한 자치구 총매출이 아니다. 공연 활동은 위치가 확보된 등록 ID 수로 실제 상연 횟수·관객 수와 다르다. 각 반경의 실제 분석 공연에 대한 공식·잠정 연결 현황은 08번 연결 검증표로 확인하며 서울 전체 목록의 연결 비율과 구분한다. 현재 좌표·경계의 과거 적용, 선택적 매출 미제공과 상권 간 공간 상관을 한계로 둔다.

업종 차이가 유의하면 '어느 업종끼리 다른가'는 사후 비교의 별도 문제다. 특정 업종은 유의하고 다른 업종은 유의하지 않다는 사실만으로 그 둘이 다르다고 결론 내리지 않는다. 새로운 업종을 포함했으므로 기존 두 업종 결과와 표준오차가 달라질 수 있다.


In [ ]:
membership = pd.concat([
    b['frame'][KEY].assign(scenario=scenario,outcome=outcome)
    for (scenario,outcome),m in model_objects.items() for b in m['blocks']
],ignore_index=True)[['scenario','outcome',*KEY]]
assert not membership.duplicated(['scenario','outcome',*KEY]).any()
outputs = {
    '09_industry_effects.csv':effects, '09_omnibus_tests.csv':joint,
    '09_model_samples.csv':pd.DataFrame(sample_rows),
    '09_eligibility.csv':pd.DataFrame(eligibility_rows),
    '09_slope_covariances.csv':pd.DataFrame(covariance_rows),
    '09_fwl_validation.csv':pd.DataFrame(verification),
    '09_activity_change_contrasts.csv':changes,
    '09_model_sample_membership.csv':membership,
    '09_activity_support.csv':activity_support}
for filename,frame in outputs.items():
    frame.to_csv(REPORTS/filename,index=False,encoding='utf-8-sig')
summary = dict(scope='종로구·마포구/공개 외식업10종/2023~2025분기',
    seoul_refresh_run_id=scope_metadata.get('seoul_refresh_run_id'),
    kopis_collection_run_id=scope_metadata['exposure_snapshot_linkage'].get('collection_run_id'),
    input_file=str(INPUT.relative_to(ROOT)).replace('\\','/'),input_sha256=input_hash,
    reference_file=str(reference_file.relative_to(ROOT)).replace('\\','/'),reference_sha256=reference_hash,
    reference_input_file=baseline_input.relative_to(ROOT).as_posix(), reference_input_sha256=baseline_input_hash,
    reference_scope='frozen_two_industry_input_and_05_results_before_refresh',
    industry_ids=list(INDUSTRIES),primary_radius_m=500,exploratory_expansion_after_two_industry_results=True,
    scenarios=[s[0] for s in SCENARIOS],outcomes=list(OUTCOMES),
    fe_systems=len(model_objects),reference_contrasts_verified=len(verification),
    primary_effect_tests=int(effects['scenario'].eq('main_500').sum()),
    primary_omnibus=joint.loc[joint['scenario'].eq('main_500')].to_dict('records'),
    fdr_significant_primary=effects.loc[effects['p_fdr_primary'].lt(.05),
        ['industry_id','industry','outcome','coefficient','p_fdr_primary']].to_dict('records'),
    causal_identification=False,python=platform.python_version(),
    output_sha256={str((REPORTS/name).relative_to(ROOT)).replace('\\','/'):
                   hashlib.sha256((REPORTS/name).read_bytes()).hexdigest() for name in outputs})
def json_finite(value):
    if isinstance(value,dict): return {k:json_finite(v) for k,v in value.items()}
    if isinstance(value,list): return [json_finite(v) for v in value]
    if isinstance(value,float) and not np.isfinite(value): return None
    return value
summary = json_finite(summary)
(REPORTS/'09_analysis_summary.json').write_text(
    json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
assert hashlib.sha256(INPUT.read_bytes()).hexdigest() == input_hash
assert hashlib.sha256(reference_file.read_bytes()).hexdigest() == reference_hash
for row in joint.loc[joint['scenario'].eq('main_500')].itertuples():
    conclusion = ('공분산 문제로 검정하지 못했다' if not np.isfinite(row.p_holm_primary) else
                  '관계의 업종 차이를 관측했다' if row.p_holm_primary<.05 else '관계의 업종 차이를 뚜렷하게 확인하지 못했다')
    display(Markdown(f"**{OUTCOMES[row.outcome]}**: {row.estimated_industries}종 동일기울기 검정 "
                     f"p={row.p_value:.4f}, Holm p={row.p_holm_primary:.4f}. {conclusion}."))
display(Markdown('신뢰구간의 크기·표본 구성·거리와 균형표본 민감도를 함께 보아야 하며 인과효과로 해석하지 않는다.'))

linkage = scope_metadata['exposure_snapshot_linkage']
display(Markdown(f"**공간 입력:** 현재 목록의 공연 {linkage['performance_ids']:,}개 중 "
    f"좌표 연결 {linkage['performance_ids_with_coordinates']:,}개, 공식 시설 ID 확인 {linkage['official_link_ids']:,}개다. "
    '좌표 확보는 과거 시설 위치 이력 또는 이름 임시 연결의 공식 검증을 뜻하지 않는다.'))


## 10. 상세 수집 반영 전후의 회귀 결과
직전 948건 상세 스냅샷의 기본 500m 결과와 같은 업종·결과변수를 대조한다. 결과 변화와 연결 신뢰도의 개선은 구분하며 유의성에 따라 사용할 스냅샷을 선택하지 않는다.

In [ ]:
previous_result_path = REPORTS / 'analysis_history/20261004_092358_9b279a/reports/09_industry_effects.csv'
previous_summary = project_json_loads((REPORTS / 'analysis_history/20261004_092358_9b279a/reports/09_analysis_summary.json').read_text(encoding='utf-8'))
assert previous_summary['output_sha256']['reports/20261005/09_industry_effects.csv'] == hashlib.sha256(previous_result_path.read_bytes()).hexdigest(), '직전 회귀 결과 해시 불일치'
previous_result = pd.read_csv(previous_result_path, dtype={'industry_id': str})
current_result = pd.read_csv(REPORTS / '09_industry_effects.csv', dtype={'industry_id': str})
comparison_keys = ['scenario', 'outcome', 'industry_id']
comparison_columns = comparison_keys + ['coefficient', 'standard_error', 'p_value']
result_revision = previous_result.loc[previous_result['scenario'].eq('main_500'), comparison_columns].merge(
    current_result.loc[current_result['scenario'].eq('main_500'), comparison_columns],
    on=comparison_keys, how='outer', validate='one_to_one', suffixes=('_previous', '_current'), indicator=True)
assert result_revision['_merge'].eq('both').all() and len(result_revision) == 20
result_revision['coefficient_change'] = result_revision['coefficient_current'] - result_revision['coefficient_previous']
result_revision.to_csv(REPORTS / '09_snapshot_model_comparison.csv', index=False, encoding='utf-8-sig')
comparison_provenance = {'previous_run_id': previous_summary['kopis_collection_run_id'],
    'current_run_id': scope_metadata['exposure_snapshot_linkage']['collection_run_id'],
    'previous_result_sha256': hashlib.sha256(previous_result_path.read_bytes()).hexdigest(),
    'current_result_sha256': hashlib.sha256((REPORTS / '09_industry_effects.csv').read_bytes()).hexdigest(),
    'comparison_sha256': hashlib.sha256((REPORTS / '09_snapshot_model_comparison.csv').read_bytes()).hexdigest(),
    'scenario': 'main_500', 'compared_industry_outcomes': len(result_revision)}
(REPORTS / '09_snapshot_model_comparison.json').write_text(json.dumps(comparison_provenance, ensure_ascii=False, indent=2), encoding='utf-8')
display(result_revision[['outcome', 'industry_id', 'coefficient_previous', 'coefficient_current', 'coefficient_change']])

**해석:** 차이가 0이면 해당 기본모형의 추정계수는 상세 수집 전후에 같다. 이는 같은 시설을 가리키던 임시 연결이 공식 ID로 확인되었을 가능성과 양립한다. 차이가 있으면 08번의 거리별 노출 변경표와 함께 읽으며 이를 실제 매출 변화나 인과효과로 해석하지 않는다.